# Heart Disease: SHAP Explanations

In this notebook we explain the final model with SHAP. We look at two views: a global view of which features matter most across all patients, and a local view of why the model gave one particular patient their prediction. The local view is what Predicta will show users.

In [3]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("../data/heart_raw.csv")
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
categorical = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(C=0.1, max_iter=1000)),
])
model.fit(X_train, y_train)
print("Test accuracy:", round(model.score(X_test, y_test), 3))

Test accuracy: 0.885
